## Zero Classification Shot

Dataset: Labelled Text Data

In [1]:
import os
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score, accuracy_score

/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ================= CONFIG =================
MODEL_NAME = "distilbert-base-uncased"  
DATA_PATH = "/home/provira/Documents/TFM/TFM/data/raw/Kaggle/csv/GoEmotions/goemotions_1.csv"
OUTPUT_DIR = "./emotion_model"
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
SEED = 42
TEST_SIZE = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ===========================================


In [3]:

# Semilla reproducible
def set_seed(seed=SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

print(f"Device: {DEVICE}")


Device: cuda


In [4]:
# ================= IMPORTS =================
from datasets import load_dataset
from transformers import AutoTokenizer

# ================= CARGAR DATASET =================
raw = load_dataset("csv", data_files=DATA_PATH)["train"]

# Columnas que NO son emociones
non_emotion_cols = {
    "text", "id", "author", "subreddit", "link_id", "parent_id",
    "created_utc", "rater_id", "example_very_unclear"
}

# Detectar columnas de emociones
label_columns = [c for c in raw.column_names if c not in non_emotion_cols]
num_labels = len(label_columns)
print("Etiquetas:", label_columns, "| Total:", num_labels)

# Asegurar que todas las etiquetas sean float (para BCEWithLogitsLoss)
def cast_labels(example):
    for c in label_columns:
        example[c] = float(example[c])
    return example

raw = raw.map(cast_labels)

# ================= SPLIT =================
split = raw.train_test_split(test_size=TEST_SIZE, seed=SEED)
train_ds = split["train"]
eval_ds = split["test"]

# ================= TOKENIZACIÓN =================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)

# ================= EMPAQUETAR LABELS =================
def pack_labels(batch):
    # batch es un diccionario de listas porque batched=True
    labels = [
        [float(batch[c][i]) for c in label_columns]
        for i in range(len(batch["text"]))
    ]
    batch["labels"] = labels
    return batch

train_ds = train_ds.map(pack_labels, batched=True)
eval_ds = eval_ds.map(pack_labels, batched=True)

# ================= FORMATO PARA TORCH =================
cols_to_return = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols_to_return)
eval_ds.set_format(type="torch", columns=cols_to_return)

# ================= COMPROBAR =================
print("Ejemplo labels:", train_ds[0]["labels"])
print("Shape:", train_ds[0]["labels"].shape)
print("Tipo:", train_ds[0]["labels"].dtype)


Etiquetas: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'] | Total: 28
Ejemplo labels: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])
Shape: torch.Size([28])
Tipo: torch.float32


In [5]:
# ================= TOKENIZACIÓN =================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)

# ================= COMBINAR LABELS =================
# Aquí juntamos todas las columnas de etiquetas en un solo tensor
def pack_labels(batch):
    batch["labels"] = [ [batch[col][i] for col in label_columns] for i in range(len(batch["text"])) ]
    return batch

train_ds = train_ds.map(pack_labels, batched=True)
eval_ds = eval_ds.map(pack_labels, batched=True)

# ================= FORMATO PARA TORCH =================
cols_to_return = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols_to_return)
eval_ds.set_format(type="torch", columns=cols_to_return)

# ================= MODELO =================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,  # número de etiquetas
    problem_type="multi_label_classification"
)

print(f"{num_labels} etiquetas listas para entrenamiento 🚀")

# ================= MÉTRICAS =================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.5).astype(int)
    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro
    }


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


28 etiquetas listas para entrenamiento 🚀


In [6]:
import transformers
print(transformers.__version__)  # should be >= 4.24

import sys
print(sys.executable)


4.56.1
/home/provira/anaconda3/envs/spark_py3.9/bin/python


In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# ================= SUBSET DEL DATASET =================
# ⚡️ 2000 ejemplos de train y 500 de eval → entrenamiento rápido
small_train = train_ds.shuffle(seed=SEED).select(range(2000))
small_eval = eval_ds.shuffle(seed=SEED).select(range(500))

# ================= MODELO LIGERO =================
MODEL_NAME = "distilbert-base-uncased"

id2label = {i: label for i, label in enumerate(label_columns)}
label2id = {label: i for i, label in enumerate(label_columns)}


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

# ================= TRAINING ARGS (CPU) =================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,                  # ⚡️ 1 epoch para probar
    per_device_train_batch_size=8,       # batch pequeño
    per_device_eval_batch_size=8,
    eval_steps=100,                      # eval cada 100 steps
    save_steps=100,                      # guarda cada 100 steps
    logging_steps=20,
    learning_rate=5e-5,
    load_best_model_at_end=False,
    save_total_limit=1,
    seed=SEED,
    no_cuda=True                         # ⚡️ fuerza CPU
)

# ================= TRAINER =================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ================= ENTRENAR =================
trainer.train()


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/transformers/training_args.py:1619: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
/tmp/ipykernel_43597/915837550.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
20,0.520800
40,0.300200
60,0.212500
80,0.172700
100,0.168500
120,0.169700
140,0.166000
160,0.162300
180,0.158400
200,0.159800


/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce GTX 1050 Ti which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/torch/cuda/__init__.py:326: UserWarning: 
NVIDIA GeForce GTX 1050 Ti with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce GTX 1050 Ti G

TrainOutput(global_step=250, training_loss=0.20724073934555054, metrics={'train_runtime': 1342.4986, 'train_samples_per_second': 1.49, 'train_steps_per_second': 0.186, 'total_flos': 66264410112000.0, 'train_loss': 0.20724073934555054, 'epoch': 1.0})

In [8]:

# ================= GUARDAR =================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modelo guardado en {OUTPUT_DIR}")

# ================= PRUEBA RÁPIDA =================
from transformers import pipeline

clf = pipeline("text-classification", model=OUTPUT_DIR, tokenizer=OUTPUT_DIR, return_all_scores=True,device=-1)

texto = "Odio madrugar pero amo ver el amanecer"
res = clf(texto)[0]

print("\n📊 Resultados:")
for r in res:
    print(f"{r['label']}: {r['score']:.2f}")

✅ Modelo guardado en ./emotion_model


Device set to use cpu



📊 Resultados:
admiration: 0.08
amusement: 0.05
anger: 0.04
annoyance: 0.05
approval: 0.09
caring: 0.03
confusion: 0.04
curiosity: 0.05
desire: 0.02
disappointment: 0.03
disapproval: 0.06
disgust: 0.02
embarrassment: 0.02
excitement: 0.02
fear: 0.02
gratitude: 0.05
grief: 0.01
joy: 0.04
love: 0.04
nervousness: 0.02
optimism: 0.05
pride: 0.01
realization: 0.04
relief: 0.01
remorse: 0.02
sadness: 0.03
surprise: 0.03
neutral: 0.28


/home/provira/anaconda3/envs/spark_py3.9/lib/python3.9/site-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(
